In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
df_labeled = pd.read_csv("Data download/data/filings_labeled.csv")
print(df_labeled.columns.tolist())

In [ ]:
DATA_DIR = Path("Data download/data")


In [ ]:
# ── CAR[0,0] — same day only ──────────────────────────────────────────────────
# AR_0 = ret_stock - (alpha + beta * ret_spy)
df_labeled['ar_0']    = df_labeled['ret_stock'] - (df_labeled['mm_alpha'] + df_labeled['mm_beta'] * df_labeled['ret_spy'])
df_labeled['car_0_0'] = df_labeled['ar_0']

# ── CAR[−1,+1] — three day window ────────────────────────────────────────────
# Need return on day -1: (price_0 - price_m1) / price_m1
# Need SPY return on day -1: (spy_price_0 - spy_price_m1) / spy_price_m1
df_labeled['ret_stock_m1'] = (df_labeled['price_0'] - df_labeled['price_m1']) / df_labeled['price_m1']
df_labeled['ret_spy_m1']   = (df_labeled['spy_price_0'] - df_labeled['spy_price_m1']) / df_labeled['spy_price_m1']

# AR on day -1
df_labeled['ar_m1'] = df_labeled['ret_stock_m1'] - (df_labeled['mm_alpha'] + df_labeled['mm_beta'] * df_labeled['ret_spy_m1'])

# CAR[-1,+1] = AR[-1] + AR[0] + AR[+1]
# AR[+1] is already embedded in car_0_1 - ar_0
df_labeled['ar_1']      = df_labeled['car_0_1'] - df_labeled['ar_0']
df_labeled['car_m1_1']  = df_labeled['ar_m1'] + df_labeled['ar_0'] + df_labeled['ar_1']

# ── Market adjusted versions ──────────────────────────────────────────────────
df_labeled['ret_stock_1'] = (df_labeled['price_p1'] - df_labeled['price_0']) / df_labeled['price_0']
df_labeled['ret_spy_1']   = (df_labeled['spy_price_p1'] - df_labeled['spy_price_0']) / df_labeled['spy_price_0']

df_labeled['car_0_0_mktadj'] = df_labeled['ret_stock'] - df_labeled['ret_spy']
df_labeled['car_m1_1_mktadj'] = (
    (df_labeled['ret_stock_m1'] - df_labeled['ret_spy_m1']) +
    (df_labeled['ret_stock']   - df_labeled['ret_spy'])     +
    (df_labeled['ret_stock_1'] - df_labeled['ret_spy_1'])
)

print("New columns computed:")
print(f"  car_0_0      : {df_labeled['car_0_0'].describe().round(4).to_dict()}")
print(f"  car_m1_1     : {df_labeled['car_m1_1'].describe().round(4).to_dict()}")

In [ ]:
# ── Load all three checkpoints ────────────────────────────────────────────────
df_lm      = pd.read_csv(DATA_DIR / "lm_checkpoint.csv")
df_finbert = pd.read_csv(DATA_DIR / "finbert_checkpoint.csv")
df_gpt     = pd.read_csv(DATA_DIR / "gpt_checkpoint.csv").dropna(subset=['gpt_score'])

# ── Use full 2,000-filing GPT-scored subsample (no train/test split) ─────────
gpt_eval = df_gpt.copy()

# ── Filter LM and FinBERT to exact same 2,000 filings as GPT ─────────────────
gpt_eval_acc = set(gpt_eval['accessionNumber'])
lm_eval      = df_lm[df_lm['accessionNumber'].isin(gpt_eval_acc)].copy()
finbert_eval = df_finbert[df_finbert['accessionNumber'].isin(gpt_eval_acc)].copy()

print(f"GPT eval     : {len(gpt_eval)}")
print(f"LM eval      : {len(lm_eval)}")
print(f"FinBERT eval : {len(finbert_eval)}")

In [ ]:
from sklearn.metrics import roc_auc_score

# Merge alternative CARs into evaluation subsamples
df_rob = df_labeled[['accessionNumber', 'car_0_0', 'car_m1_1', 
                       'car_0_0_mktadj', 'car_m1_1_mktadj']]

lm_rob      = lm_eval.merge(df_rob, on='accessionNumber')
finbert_rob = finbert_eval.merge(df_rob, on='accessionNumber')
gpt_rob     = gpt_eval.merge(df_rob, on='accessionNumber')

print("=== Robustness Check 1: Alternative Event Windows ===")
print("(Primary metric: AUC on evaluation subsample, N=2,000)\n")

windows = {
    'CAR[0,0]'   : 'car_0_0',
    'CAR[0,+1]'  : 'car_0_1',   # primary — for reference
    'CAR[-1,+1]' : 'car_m1_1',
}

print(f"{'Window':<14} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 50)

for window_name, car_col in windows.items():
    # Recompute downside label at 25th percentile for each window
    threshold    = df_labeled[car_col].quantile(0.25)
    df_labeled[f'downside_{car_col}'] = (df_labeled[car_col] < threshold).astype(int)

    lm_rob      = lm_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')
    finbert_rob = finbert_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')
    gpt_rob     = gpt_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')

    auc_lm  = roc_auc_score(lm_rob[f'downside_{car_col}'],      lm_rob['lm_neg_prop'])
    auc_fb  = roc_auc_score(finbert_rob[f'downside_{car_col}'], finbert_rob['finbert_neg_mean'])
    auc_gpt = roc_auc_score(gpt_rob[f'downside_{car_col}'],    -gpt_rob['gpt_score'])

    primary = " ← primary" if car_col == 'car_0_1' else ""
    print(f"{window_name:<14} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}{primary}")

In [ ]:
print("=== Robustness Check 2: Market-Adjusted Returns ===\n")

# Compute market-adjusted downside label
threshold_mktadj = df_labeled['car_0_1_mktadj'].quantile(0.25)
df_labeled['downside_mktadj'] = (df_labeled['car_0_1_mktadj'] < threshold_mktadj).astype(int)

print(f"{'Return model':<28} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 64)

# Primary: market model — use existing downside column already in evaluation subsamples
auc_lm  = roc_auc_score(lm_eval['downside'],      lm_eval['lm_neg_prop'])
auc_fb  = roc_auc_score(finbert_eval['downside'],  finbert_eval['finbert_neg_mean'])
auc_gpt = roc_auc_score(gpt_eval['downside'],     -gpt_eval['gpt_score'])
print(f"{'CAR[0,+1] market model':<28} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f} ← primary")

# Robustness: market-adjusted — merge only the new mktadj downside label
mktadj_labels = df_labeled[['accessionNumber', 'downside_mktadj']]
lm_rob      = lm_eval.merge(mktadj_labels, on='accessionNumber')
finbert_rob = finbert_eval.merge(mktadj_labels, on='accessionNumber')
gpt_rob     = gpt_eval.merge(mktadj_labels, on='accessionNumber')

auc_lm  = roc_auc_score(lm_rob['downside_mktadj'],      lm_rob['lm_neg_prop'])
auc_fb  = roc_auc_score(finbert_rob['downside_mktadj'],  finbert_rob['finbert_neg_mean'])
auc_gpt = roc_auc_score(gpt_rob['downside_mktadj'],     -gpt_rob['gpt_score'])
print(f"{'CAR[0,+1] market adj':<28} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}")

In [ ]:
print("=== Robustness Check 3: Downside Threshold Sensitivity ===\n")

thresholds = {
    'Percentile 25% (−3.90%)'  : df_labeled['car_0_1'].quantile(0.25),   # primary
    'Fixed −1%'                : -0.01,
    'Fixed −3%'                : -0.03,
}

print(f"{'Threshold':<28} {'Downside N':>12} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 76)

for threshold_name, cutoff in thresholds.items():
    df_labeled['downside_thresh'] = (df_labeled['car_0_1'] < cutoff).astype(int)

    lm_rob      = lm_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')
    finbert_rob = finbert_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')
    gpt_rob     = gpt_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')

    n_down = lm_rob['downside_thresh'].sum()

    try:
        auc_lm  = roc_auc_score(lm_rob['downside_thresh'],      lm_rob['lm_neg_prop'])
        auc_fb  = roc_auc_score(finbert_rob['downside_thresh'],  finbert_rob['finbert_neg_mean'])
        auc_gpt = roc_auc_score(gpt_rob['downside_thresh'],     -gpt_rob['gpt_score'])
        primary = " ← primary" if 'Percentile' in threshold_name else ""
        print(f"{threshold_name:<28} {n_down:>12} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}{primary}")
    except ValueError as e:
        print(f"{threshold_name:<28} skipped — {e}")